# Zhiheng – Exploratory Test Notebook

Personal sandbox for testing model components, data pipelines, and
running quick experiments without touching the main training workflow.

In [ ]:
import sys
import os

# Add repo root to path when running from the experiments/ directory
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

import torch
import numpy as np
import matplotlib.pyplot as plt

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Smoke-test: CLIP Encoder

In [ ]:
from models.clip_encoder import CLIPEncoder

encoder = CLIPEncoder(projection_dim=128, freeze_clip=True).to(device)
encoder.eval()

# Dummy inputs
dummy_text = ['Battlecry: Deal 3 damage.', 'Taunt. Divine Shield.']
inputs = encoder.preprocess(texts=dummy_text)
input_ids = inputs['input_ids'].to(device)
attention_mask = inputs['attention_mask'].to(device)

with torch.no_grad():
    text_emb = encoder.encode_text(input_ids, attention_mask)

print('Text embedding shape:', text_emb.shape)  # Expected: (2, 128)

## Smoke-test: GNN Model

In [ ]:
from models.gnn_model import GNNModel

gnn = GNNModel(in_channels=128 + 3 + 18, hidden_channels=64, out_channels=32).to(device)
gnn.eval()

# Dummy node features (5 nodes)
x = torch.randn(5, 128 + 3 + 18).to(device)
# Dummy edges (fully connected 5-node graph)
edge_index = torch.tensor(
    [[i for i in range(5) for j in range(5) if i != j],
     [j for i in range(5) for j in range(5) if i != j]],
    dtype=torch.long
).to(device)

with torch.no_grad():
    out = gnn(x, edge_index)

print('GNN output shape:', out.shape)  # Expected: (5, 32)

## Smoke-test: Preprocessing Utilities

In [ ]:
from utils.preprocessing import clean_card_text, normalise_card_stats, card_to_one_hot

sample_card = {
    'name': 'Fireball',
    'type': 'SPELL',
    'cardClass': 'MAGE',
    'cost': 4,
    'text': '<b>Deal 6 damage.</b>',
}

print('Cleaned text :', clean_card_text(sample_card['text']))
print('Normalised stats:', normalise_card_stats(sample_card))
print('One-hot shape:', card_to_one_hot(sample_card).shape)  # Expected: torch.Size([18])

## Embedding Similarity Exploration

In [ ]:
import torch.nn.functional as F

card_texts = [
    'Battlecry: Deal 3 damage to a minion.',
    'Taunt. Battlecry: Gain +2/+2.',
    'Deathrattle: Summon a 2/2 Ghost.',
    'Freeze all enemy minions.',
    'Lifesteal. Battlecry: Restore 5 Health.',
]

inputs = encoder.preprocess(texts=card_texts)
input_ids = inputs['input_ids'].to(device)
attention_mask = inputs['attention_mask'].to(device)

with torch.no_grad():
    embeddings = encoder.encode_text(input_ids, attention_mask)
    embeddings = F.normalize(embeddings, dim=-1)

sim_matrix = (embeddings @ embeddings.T).cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(card_texts)))
ax.set_yticks(range(len(card_texts)))
short_labels = [t[:25] + '…' if len(t) > 25 else t for t in card_texts]
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(short_labels, fontsize=7)
plt.colorbar(im, ax=ax)
ax.set_title('Cosine Similarity of Card Text Embeddings')
plt.tight_layout()
plt.show()